**Preditor baseado no PDF de AG**

In [8]:
#bibliotecas necessarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.feature_selection import mutual_info_classif, f_classif

In [9]:
# configurações e dados iniciais
#o AG possui componentes aleatórios, então essa semente permite controlar a aleatoriedade para manter os mesmos numeros no decorrer do projeto
SEMENTE = 42

NUM_LAGS = 7 #teremos 7 lags que serão usados nas features

PERIODO_WMA = 30 #30 dias

# parametros do Algoritmo Genético
TAMANHO_POPULACAO = 100
TAXA_MUTACAO = 0.001

# numero de gerações
GERACOES_FILTER = 10000
GERACOES_WRAPPER = 1000

In [10]:
# series utilizadas no experimento E1 

bitcoin = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Number of Transactions",
    "Transaction Fees",
    "Cost per Transaction",
    "Average Hash Rate"
]

commodities = [
    "Crude Oil",
    "Gold",
    "Silver",
    "Coffee",
    "Heating Oil",
    "Natural Gas",
    "Platinum",
    "Palladium",
    "Copper",
    "Cocoa",
    "Sugar",
    "Oat"
]

indices = [
    "S&P 500",
    "NASDAQ",
    "DAX",
    "KOSPI"
]

# todas as séries utilizadas no experimento E1
features_base = bitcoin + commodities + indices

print("Número de variáveis:", len(features_base))
print(features_base)

Número de variáveis: 25
['Open', 'High', 'Low', 'Close', 'Volume', 'Number of Transactions', 'Transaction Fees', 'Cost per Transaction', 'Average Hash Rate', 'Crude Oil', 'Gold', 'Silver', 'Coffee', 'Heating Oil', 'Natural Gas', 'Platinum', 'Palladium', 'Copper', 'Cocoa', 'Sugar', 'Oat', 'S&P 500', 'NASDAQ', 'DAX', 'KOSPI']


In [12]:
# nova janela temporal usada no E1 (vou usar datas mais recentes)

DATA_INICIO = "2021-01-01"
DATA_FIM = "2026-01-01"


# Intervalo I1
I1_TREINO_INICIO = "2021-01-01"
I1_TREINO_FIM = "2024-12-31"

I1_TESTE_INICIO = "2025-01-01"
I1_TESTE_FIM = "2026-01-01"


# Intervalo I2
I2_TREINO_INICIO = "2021-01-01"
I2_TREINO_FIM = "2024-06-30"

I2_TESTE_INICIO = "2024-07-01"
I2_TESTE_FIM = "2026-01-01"


print("Janela total:")
print(f"{DATA_INICIO} até {DATA_FIM}")

print("\nI1:")
print(f"Treinamento: {I1_TREINO_INICIO} até {I1_TREINO_FIM}")
print(f"Teste:       {I1_TESTE_INICIO} até {I1_TESTE_FIM}")

print("\nI2:")
print(f"Treinamento: {I2_TREINO_INICIO} até {I2_TREINO_FIM}")
print(f"Teste:       {I2_TESTE_INICIO} até {I2_TESTE_FIM}")

#I1 → 80% treinamento / 20% teste
#I2 → 75% treinamento / 25% teste

Janela total:
2021-01-01 até 2026-01-01

I1:
Treinamento: 2021-01-01 até 2024-12-31
Teste:       2025-01-01 até 2026-01-01

I2:
Treinamento: 2021-01-01 até 2024-06-30
Teste:       2024-07-01 até 2026-01-01


In [17]:
import yfinance as yf #yahoo finance

In [19]:
# baixar os dados diários do Bitcoin

dados_btc = yf.download(
    "BTC-USD", #Bitcoin em dolares
    start=DATA_INICIO,
    end=DATA_FIM,
    interval="1d", #intervalo de 1 dia
    auto_adjust=False
)

print("Quantidade de linhas:", len(dados_btc))
print("\nPrimeiras linhas:")
display(dados_btc.head())

print("\nÚltimas linhas:")
display(dados_btc.tail())

[*********************100%***********************]  1 of 1 completed

Quantidade de linhas: 1826

Primeiras linhas:


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,,
2021-01-01,29374.152344,29374.152344,29600.626953,28803.585938,28994.009766,40730301359
2021-01-02,32127.267578,32127.267578,33155.117188,29091.181641,29376.455078,67865420765
2021-01-03,32782.023438,32782.023438,34608.558594,32052.316406,32129.408203,78665235202
2021-01-04,31971.914062,31971.914062,33440.218750,28722.755859,32810.949219,81163475344
2021-01-05,33992.429688,33992.429688,34437.589844,30221.187500,31977.041016,67547324782



Últimas linhas:


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,,
2025-12-27,87802.156250,87802.156250,87874.781250,87182.976562,87301.429688,13741199310
2025-12-28,87835.835938,87835.835938,87986.890625,87394.953125,87799.343750,15156557929
2025-12-29,87138.140625,87138.140625,90299.156250,86717.914062,87835.789062,48411625849
2025-12-30,88430.132812,88430.132812,89297.937500,86735.546875,87134.351562,35586356225
2025-12-31,87508.828125,87508.828125,89080.289062,87130.562500,88429.585938,33830210616


In [20]:
# remove o nível "Ticker" das colunas

dados_btc.columns = dados_btc.columns.get_level_values(0)

# mantem apenas as variáveis utilizadas no experimento
dados_btc = dados_btc[
    ["Open", "High", "Low", "Close", "Volume"]
].copy()

# garante que o índice seja uma data
dados_btc.index = pd.to_datetime(dados_btc.index)

print("Colunas:")
print(dados_btc.columns.tolist())

print("\nFormato dos dados:")
print(dados_btc.shape)

display(dados_btc.head())

Colunas:
['Open', 'High', 'Low', 'Close', 'Volume']

Formato dos dados:
(1826, 5)


Price,Open,High,Low,Close,Volume
Date,,,,,
2021-01-01,28994.009766,29600.626953,28803.585938,29374.152344,40730301359
2021-01-02,29376.455078,33155.117188,29091.181641,32127.267578,67865420765
2021-01-03,32129.408203,34608.558594,32052.316406,32782.023438,78665235202
2021-01-04,32810.949219,33440.218750,28722.755859,31971.914062,81163475344
2021-01-05,31977.041016,34437.589844,30221.187500,33992.429688,67547324782


In [26]:
!pip install nasdaq-data-link


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\victo\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [27]:
import nasdaqdatalink

print("Biblioteca Nasdaq Data Link carregada com sucesso!")

Biblioteca Nasdaq Data Link carregada com sucesso!


In [29]:
# Teste de acesso ao Nasdaq Data Link

try:
    dados_teste = nasdaqdatalink.get("BCHAIN/ATRCT")
    print("Consulta realizada com sucesso!")
    display(dados_teste.head())
except Exception as erro:
    print("Não foi possível acessar a série.")
    print("\nErro:")
    print(erro)

Não foi possível acessar a série.

Erro:
(Status 403) Something went wrong. Please try again. If you continue to have problems, please contact us at connect@data.nasdaq.com.


In [31]:
import pandas as pd

url = "https://api.blockchain.info/charts/n-transactions-total?timespan=all&format=json"

teste = pd.read_json(url)

print("Dados obtidos com sucesso!")
display(teste.head())

Dados obtidos com sucesso!


,status,name,unit,period,description,values
0,ok,Total Number of Transactions,Transactions,day,Total Number of transactions,"{'x': 1230940800, 'y': 1.0}"
1,ok,Total Number of Transactions,Transactions,day,Total Number of transactions,"{'x': 1231718400, 'y': 170.0}"
2,ok,Total Number of Transactions,Transactions,day,Total Number of transactions,"{'x': 1232064000, 'y': 658.0}"
3,ok,Total Number of Transactions,Transactions,day,Total Number of transactions,"{'x': 1232409600, 'y': 1102.0}"
4,ok,Total Number of Transactions,Transactions,day,Total Number of transactions,"{'x': 1232755200, 'y': 1498.0}"


In [33]:
# URLs das séries de blockchain do Bitcoin

urls_blockchain = {
    "Number of Transactions":
        "https://api.blockchain.info/charts/n-transactions-total?timespan=all&format=json",

    "Transaction Fees":
        "https://api.blockchain.info/charts/transaction-fees?timespan=all&format=json",

    "Cost per Transaction":
        "https://api.blockchain.info/charts/cost-per-transaction?timespan=all&format=json",

    "Average Hash Rate":
        "https://api.blockchain.info/charts/hash-rate?timespan=all&format=json"
}

dados_blockchain = {}

for nome, url in urls_blockchain.items():
    resposta = pd.read_json(url)

    valores = pd.json_normalize(resposta["values"])

    valores["Date"] = pd.to_datetime(
        valores["x"],
        unit="s"
    )

    valores = valores.set_index("Date")

    valores = valores[["y"]]
    valores.columns = [nome]

    dados_blockchain[nome] = valores

print("Séries obtidas:")

for nome, dados in dados_blockchain.items():
    print(f"- {nome}: {len(dados)} registros")

display(dados_blockchain["Number of Transactions"].head())

Séries obtidas:
- Number of Transactions: 1619 registros
- Transaction Fees: 1617 registros
- Cost per Transaction: 1618 registros
- Average Hash Rate: 1621 registros


,Number of Transactions
Date,
2009-01-03,1.0
2009-01-12,170.0
2009-01-16,658.0
2009-01-20,1102.0
2009-01-24,1498.0


In [34]:
# Seleciona somente o período que estamos usando no projeto
dados_blockchain_periodo = {}

for nome, dados in dados_blockchain.items():
    dados = dados.loc[
        DATA_INICIO:DATA_FIM
    ].copy()

    dados_blockchain_periodo[nome] = dados


# Junta as 4 séries de blockchain
dados_blockchain_final = pd.concat(
    dados_blockchain_periodo.values(),
    axis=1
)


# Junta com os dados de mercado do Bitcoin
dados_bitcoin = dados_btc.join(
    dados_blockchain_final,
    how="left"
)


print("Formato antes do preenchimento:")
print(dados_bitcoin.shape)

print("\nValores ausentes por variável:")
print(dados_bitcoin.isna().sum())

display(dados_bitcoin.head())

Formato antes do preenchimento:
(1826, 9)

Valores ausentes por variável:
Open                         0
High                         0
Low                          0
Close                        0
Volume                       0
Number of Transactions    1370
Transaction Fees          1370
Cost per Transaction      1369
Average Hash Rate         1371
dtype: int64


C:\Users\victo\AppData\Local\Temp\ipykernel_34076\3004917288.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  dados_blockchain_final = pd.concat(


,Open,High,Low,Close,Volume,Number of Transactions,Transaction Fees,Cost per Transaction,Average Hash Rate
Date,,,,,,,,,
2021-01-01,28994.009766,29600.626953,28803.585938,29374.152344,40730301359,601577750.0,NaN,NaN,NaN
2021-01-02,29376.455078,33155.117188,29091.181641,32127.267578,67865420765,NaN,79.699067,107.041738,NaN
2021-01-03,32129.408203,34608.558594,32052.316406,32782.023438,78665235202,NaN,NaN,NaN,NaN
2021-01-04,32810.949219,33440.218750,28722.755859,31971.914062,81163475344,NaN,NaN,NaN,1.599542e+08
2021-01-05,31977.041016,34437.589844,30221.187500,33992.429688,67547324782,602865747.0,NaN,NaN,NaN


In [35]:
# Preenchimento dos dados ausentes
# Seguindo a metodologia do artigo: forward-fill (ffill)

dados_bitcoin = dados_bitcoin.ffill()

print("Valores ausentes após o ffill:")
print(dados_bitcoin.isna().sum())

print("\nFormato final:")
print(dados_bitcoin.shape)

display(dados_bitcoin.head())

Valores ausentes após o ffill:
Open                      0
High                      0
Low                       0
Close                     0
Volume                    0
Number of Transactions    0
Transaction Fees          1
Cost per Transaction      1
Average Hash Rate         3
dtype: int64

Formato final:
(1826, 9)


,Open,High,Low,Close,Volume,Number of Transactions,Transaction Fees,Cost per Transaction,Average Hash Rate
Date,,,,,,,,,
2021-01-01,28994.009766,29600.626953,28803.585938,29374.152344,40730301359,601577750.0,NaN,NaN,NaN
2021-01-02,29376.455078,33155.117188,29091.181641,32127.267578,67865420765,601577750.0,79.699067,107.041738,NaN
2021-01-03,32129.408203,34608.558594,32052.316406,32782.023438,78665235202,601577750.0,79.699067,107.041738,NaN
2021-01-04,32810.949219,33440.218750,28722.755859,31971.914062,81163475344,601577750.0,79.699067,107.041738,1.599542e+08
2021-01-05,31977.041016,34437.589844,30221.187500,33992.429688,67547324782,602865747.0,79.699067,107.041738,1.599542e+08
